 # 📍 Address Extraction for Geocoding (2024 Voterfile)

This Python script retrieves the most recent 2024 voterfile snapshot, selecting only the address-related columns (street, city, state, ZIP).
The goal is to prepare these records for geocoding via a self-hosted Nominatim instance running on a local server.
The latitude and longitude results will later be used for spatial analysis on Florida’s voterfile data.

# Nominatim Connectivity Test (Hosted on Home Server)
This prototype script is used to verify that the Nominatim geocoding service—hosted locally on my home server—is running correctly and responding to basic address queries. It’s a quick way to confirm API connectivity and geocoding functionality before launching full-scale batch processing.

In [ ]:
import requests

# --- Prototype Test: Verifying Nominatim API Connectivity ---
# This script is used to quickly test if the Nominatim container is up and responding to requests.
# Useful for debugging address lookup without involving the full ETL pipeline.
# You can change the `address` below to check different test cases.

address = "2758   BELLVIEW AVE "
url = f"http://192.168.86.36:8080/search"
params = {
    "q": address,
    "format": "json",
    "limit": 1
}

response = requests.get(url, params=params)
data = response.json()

if data:
    print("Latitude:", data[0]["lat"])
    print("Longitude:", data[0]["lon"])
else:
    print("No match found.")

# Step 1: Retrieve Voter Address from Voterfile Database

This step queries the voterfile database to extract key address fields (street, city, state, ZIP) for all registered voters.  
The output will be used in subsequent geocoding steps to attach geographic coordinates for spatial analysis.

In [1]:
import psycopg2

# Database connection parameters 
db_name = "fl_election"
db_user = "postgres"
db_password = "1434"
db_host = "192.168.86.36" # Change to server's IP when acessing remotely
db_port = "5432"

def connect_to_db():
    """ Establish a connection to PostgreSQL and return the connection & cursor. """
    
    try:
        print("Attempting to connect...")
        conn = psycopg2.connect(
            dbname = db_name,
            user = db_user,
            password = db_password,
            host = db_host,
            port = db_port
        )
        cur = conn.cursor()
        print("Connected to PostgreSQL")
        return conn, cur
    except Exception as e:
        print(f"Database connection error: {e}")
        return None, None
    
# Call the function to test connection
conn, cur = connect_to_db()

Attempting to connect...
Connected to PostgreSQL


In [ ]:
import pandas as pd

query = "select residence_address_1 street, residence_city_usps city, residence_zipcode zip_code, 'FL' AS state from voterfile.election_detail_2024"

df = pd.read_sql(query, conn)

/var/folders/h3/qc0834sx6gjb7174_439hbkc0000gn/T/ipykernel_47072/889405858.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


# Step 2: Create Normalized `formatted_address` Column

This step constructs a unified `formatted_address` field from raw voterfile columns (street, city, state, and zip).  
Normalization includes:
- Stripping extra whitespace and applying consistent casing
- Converting full directional words (e.g., "Southwest") to abbreviations ("SW")
- Standardizing common street suffixes (e.g., "Avenue" → "Ave", "Street" → "St")

This ensures clean, consistent input for geocoding via Nominatim.

In [ ]:
import re 
import pandas as pd

def normalize_address(row):

    # lowercase + strip white space
    street = str(row['street']).strip().lower()
    street = re.sub(r'\s+', ' ', street)  # normalize extra spaces
    city = str(row['city']).strip().title()
    state = str(row['state']).strip().upper()
    zip_code = str(row['zip_code']).strip().upper().zfill(5)
    
    directionals = {
        'northwest' : 'nw',
        'northwest' : 'ne',
        'southwest' : 'sw',
        'southeast' : 'se',
        'north' : 'n',
        'south' : 's',
        'east' : 'e',
        'west' : 'w',
    }

    for full, abbr in directionals.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)

    # Normaization street suffixes 
    suffixes = {
        'street' : 'st',
        'avenue' : 'ave',
        'boulevard' : 'blvd', 
        'road' : 'rd', 
        'drive' : 'dr', 
        'lane' : 'ln', 
        'court' : 'ct'
     }
    for full, abbr in suffixes.items():
        street = re.sub(r'\b' + full + r'\b', abbr, street)
    
    # combined the final address string
    full_address = f"{street.title()}, {city}, {state} {zip_code}"
    return full_address
    

df['formatted_address'] = df.apply(normalize_address, axis=1)


# Step 3: Create SQLite Database for Caching Geocoding Results

To handle the large size of the voterfile (~8 million rows), we use a local SQLite database to **cache geocoding results**.  
This prevents re-querying the same address multiple times and safeguards progress in case of interruptions (e.g., network errors, power loss).

The function below allows flexible creation of a `.db` file and table by specifying the table name and database name.

This cache will:
- Save lat/lon coordinates for previously queried addresses
- Track geocoding status (`success`, `no_match`, or `error`)
- Speed up repeated runs and make the process fault-tolerant

In [5]:
import sqlite3

def initialize_cache_db(table_name: str, db_name: str):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            address TEXT PRIMARY KEY,
            lat REAL,
            lon REAL,
            status TEXT
        )
    """)

    conn.commit()
    conn.close()
    print(f"Table '{table_name}' created or already exists in database '{db_name}'!")

# Example usage:
# initialize_cache_db("geocode_cache", "geocache_test.db")
# initialize_cache_db("geocode_cache_rerun", "geocache_rerun.db")
initialize_cache_db("geocode_cache_rerun_threads", "geocache_rerun_threads.db")

Table 'geocode_cache_rerun_threads' created or already exists in database 'geocache_rerun_threads.db'!


# Step 4: Geocode Addresses with Local Nominatim & Cache Results

This step defines a Python function that takes a single formatted address, sends a request to your locally hosted Nominatim instance, and caches the result in a custom SQLite database and table.

Parameters:
	•	address (str): A formatted address string, e.g., "1222 SW 103rd Ln, Miami, FL 33196"
	•	db_name (str): Name of the SQLite database file (e.g., "geocache_test.db")
	•	table_name (str): Name of the table where results will be cached (e.g., "geocode_cache")

In [2]:
import requests
import sqlite3

def geocode_address_cached(address, db_name, table_name, base_url="http://192.168.86.36:8080"):
    """Geocode a single address using local Nominatim instance and cache results in a custom DB/table."""
    if not address or pd.isna(address) or address.strip() == "":
        return None, None, "invalid_input"
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    # Check if address is already in the cache
    cur.execute(f"SELECT lat, lon, status FROM {table_name} WHERE address = ?", (address,))
    row = cur.fetchone()

    if row:
        conn.close()
        return row[0], row[1], row[2]

    # Try geocoding via Nominatim
    try:
        response = requests.get(
            f"{base_url}/search",
            params={"q": address, "format": "json", "limit": 1},
            timeout=5
        )
        response.raise_for_status()
        data = response.json()

        if data:
            lat = float(data[0]["lat"])
            lon = float(data[0]["lon"])
            status = "success"
        else:
            lat, lon = None, None
            status = "no_match"

    except Exception as e:
        print(f"Error: {e} for {address}")
        lat, lon = None, None
        status = "error"

    # Cache the result regardless of outcome
    cur.execute(
        f"INSERT INTO {table_name} (address, lat, lon, status) VALUES (?, ?, ?, ?)",
        (address, lat, lon, status)
    )
    conn.commit()
    conn.close()

    return lat, lon, status

# Step 5: Apply Geocoding Function to Entire DataFrame

This method ensures you’re only querying uncached addresses and logging every result for future use.

📌 Warning: 8 million rows will take multiple days. Parallelism is number 1 prioritiy for v2 

What This Does:
	•	Progress tracking: tqdm.pandas() adds a real-time progress bar so you can monitor the status of the geocoding process.
	•	Row-wise geocoding: Each row’s formatted_address is passed to the geocoding function.
	•	Caching results: Lat, lon, and status (e.g. success, no_match, or error) are returned and stored in the DataFrame.
	•	Fault tolerance: If the script stops or fails, previously geocoded rows are already saved in your .db file and won’t be queried again.


In [ ]:
from tqdm import tqdm

tqdm.pandas()  # Enables progress bar for pandas apply

# Define your database and table names
db_name = "geocache_test.db"
table_name = "geocode_cache"

# Use lambda to pass additional arguments into the geocode function
df['lat'], df['lon'], df['status'] = zip(*df['formatted_address'].progress_apply(
    lambda addr: geocode_address_cached(addr, db_name, table_name)
))

# Step 6: Load Cached Geocode Results into a DataFrame

Now that we’ve cached millions of address lookups into an SQLite database, we want a fast and reusable way to reload those results for inspection or reuse without rerunning the full geocoding process.

In [3]:
import sqlite3
import pandas as pd

def load_table_as_df(db_name, table_name):
    """
    Load a table from a SQLite database into a pandas DataFrame.
    
    Args:
        db_name (str): The path to the SQLite database file.
        table_name (str): The name of the table to load.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    conn = sqlite3.connect(db_name)
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    
    print(f"Preview of '{table_name}' from '{db_name}':")
    print(df.head())
    
    return df

In [ ]:
# Initalize first db 
df_cache_1 = load_table_as_df("geocache_test.db", "geocode_cache")

In [21]:
from sqlalchemy import create_engine

# Construct the SQLAlchemy connection string
connection_string = f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

# Create the engine
engine = create_engine(connection_string)





In [24]:
df_cache_1.to_sql(
    "voterfile_122024",      # Table name
    con=engine,
    schema="geo",            # Your schema
    if_exists="replace",     # or "append"
    index=False
)

1